# Multi-channel CNN + LSTM cho Vietnamese Sentiment Classification
Bài toán mặc định: **sentiment classification tiếng Việt** với 3 nhãn:

- `0`: negative
- `1`: neutral
- `2`: positive

Kiến trúc:

```text
Input sentence
→ tokenize + pad/truncate
→ embedding 200d
→ CNN channel: Conv1D kernel 3, 5, 7 + global max pooling
→ LSTM channel: LSTM hidden 128
→ concat(CNN features, LSTM feature)
→ Dense 200 + sigmoid
→ Dropout 0.2
→ Linear classifier
```

## 0. Mục tiêu bài làm

Notebook này thực hiện bài toán **Vietnamese sentiment classification** trên dataset **UIT-VSFC**. Mô hình chính là kiến trúc **Multi-channel CNN + LSTM**, được xây dựng theo tinh thần của bài báo *Multi-channel LSTM-CNN model for Vietnamese sentiment analysis* [*Multi-channel LSTM-CNN model for Vietnamese sentiment analysis*](https://www.researchgate.net/publication/321259272_Multi-channel_LSTM-CNN_model_for_Vietnamese_sentiment_analysis).

Các bước chính:

1. Tải và tiền xử lý dữ liệu tiếng Việt.
2. Xây dựng vocabulary và DataLoader.
3. Huấn luyện mô hình Multi-channel CNN + LSTM.
4. Đánh giá trên tập test.
5. So sánh với một số baseline: TF-IDF, CNN-only và LSTM-only.
6. Tổng hợp kết quả và nhận xét.


## 1. Cài đặt thư viện

In [1]:
# !pip -q install torch datasets scikit-learn tqdm numpy pandas

## 2. Import và cấu hình

In [2]:
from __future__ import annotations

import json
import os
import random
import re
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"

@dataclass
class TrainConfig:
    task: str = "sentiment"          # "sentiment" hoặc "topic"
    max_features: int = 21540
    max_len: int = 400
    embedding_dim: int = 200
    num_filters: int = 150
    kernel_sizes: Tuple[int, ...] = (3, 5, 7)
    lstm_hidden: int = 128
    dense_hidden: int = 200
    dropout: float = 0.2
    batch_size: int = 64
    epochs: int = 14
    lr: float = 0.002                
    seed: int = 1337
    use_class_weights: bool = False  
    output_dir: str = "outputs_sentiment_cnn_lstm"

cfg = TrainConfig()

cfg

TrainConfig(task='sentiment', max_features=21540, max_len=400, embedding_dim=200, num_filters=150, kernel_sizes=(3, 5, 7), lstm_hidden=128, dense_hidden=200, dropout=0.2, batch_size=64, epochs=14, lr=0.002, seed=1337, use_class_weights=False, output_dir='outputs_sentiment_cnn_lstm')

## 3. Reproducibility và tokenizer đơn giản cho tiếng Việt

In [3]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def basic_vi_tokenize(text: str) -> List[str]:
    text = text.lower().strip()
    text = re.sub(r"([^\w\s])", r" \1 ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split() if text else []


def build_vocab(texts: Sequence[str], max_features: int, min_freq: int = 1) -> Dict[str, int]:
    counter: Counter[str] = Counter()
    for text in texts:
        counter.update(basic_vi_tokenize(text))

    vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
    for token, freq in counter.most_common(max_features - len(vocab)):
        if freq >= min_freq:
            vocab[token] = len(vocab)
    return vocab


def encode_text(text: str, vocab: Dict[str, int], max_len: int) -> List[int]:
    ids = [vocab.get(tok, vocab[UNK_TOKEN]) for tok in basic_vi_tokenize(text)]
    ids = ids[:max_len]
    if len(ids) < max_len:
        ids += [vocab[PAD_TOKEN]] * (max_len - len(ids))
    return ids

seed_everything(cfg.seed)
print(basic_vi_tokenize("Thầy Huy và thầy Tùng dạy rất hay, nhiệt tình!"))

['thầy', 'huy', 'và', 'thầy', 'tùng', 'dạy', 'rất', 'hay', ',', 'nhiệt', 'tình', '!']


## 4. Tải dataset UIT-VSFC

In [4]:
from huggingface_hub import hf_hub_download

repo_id = "uitnlp/vietnamese_students_feedback"
revision = "refs/convert/parquet"

data_files = {
    "train": hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        revision=revision,
        filename="default/train/0000.parquet",
    ),
    "validation": hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        revision=revision,
        filename="default/validation/0000.parquet",
    ),
    "test": hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        revision=revision,
        filename="default/test/0000.parquet",
    ),
}

ds = load_dataset("parquet", data_files=data_files)
ds

default/train/0000.parquet:   0%|          | 0.00/475k [00:00<?, ?B/s]

default/validation/0000.parquet:   0%|          | 0.00/63.3k [00:00<?, ?B/s]

default/test/0000.parquet:   0%|          | 0.00/134k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'sentiment', 'topic'],
        num_rows: 11426
    })
    validation: Dataset({
        features: ['sentence', 'sentiment', 'topic'],
        num_rows: 1583
    })
    test: Dataset({
        features: ['sentence', 'sentiment', 'topic'],
        num_rows: 3166
    })
})

In [5]:
for i in range(5):
    print(ds["train"][i])

{'sentence': 'slide giáo trình đầy đủ .', 'sentiment': 2, 'topic': 1}
{'sentence': 'nhiệt tình giảng dạy , gần gũi với sinh viên .', 'sentiment': 2, 'topic': 0}
{'sentence': 'đi học đầy đủ full điểm chuyên cần .', 'sentiment': 0, 'topic': 1}
{'sentence': 'chưa áp dụng công nghệ thông tin và các thiết bị hỗ trợ cho việc giảng dạy .', 'sentiment': 0, 'topic': 0}
{'sentence': 'thầy giảng bài hay , có nhiều bài tập ví dụ ngay trên lớp .', 'sentiment': 2, 'topic': 0}


In [6]:
# sentiment: 0 negative, 1 neutral, 2 positive
# topic: 0 lecturer, 1 training_program, 2 facility, 3 others

def get_split(split: str, task: str):
    texts = list(ds[split]["sentence"])
    labels = [int(x) for x in ds[split][task]]
    return texts, labels

train_texts, train_labels = get_split("train", cfg.task)
val_texts, val_labels = get_split("validation", cfg.task)
test_texts, test_labels = get_split("test", cfg.task)

print("train:", len(train_texts))
print("validation:", len(val_texts))
print("test:", len(test_texts))
print("label distribution train:", Counter(train_labels))

train: 11426
validation: 1583
test: 3166
label distribution train: Counter({2: 5643, 0: 5325, 1: 458})


## 5. Dataset, vocab và DataLoader

In [7]:
class TextClassificationDataset(Dataset):
    def __init__(self, texts: Sequence[str], labels: Sequence[int], vocab: Dict[str, int], max_len: int):
        self.texts = list(texts)
        self.labels = list(labels)
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int):
        x = encode_text(self.texts[idx], self.vocab, self.max_len)
        y = int(self.labels[idx])
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


vocab = build_vocab(train_texts, max_features=cfg.max_features)
print("vocab_size:", len(vocab))

train_ds = TextClassificationDataset(train_texts, train_labels, vocab, cfg.max_len)
val_ds = TextClassificationDataset(val_texts, val_labels, vocab, cfg.max_len)
test_ds = TextClassificationDataset(test_texts, test_labels, vocab, cfg.max_len)


def make_loaders(batch_size: int = None, seed: int = None):
    if batch_size is None:
        batch_size = cfg.batch_size
    if seed is None:
        seed = cfg.seed

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        generator=generator,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )
    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = make_loaders(cfg.batch_size, cfg.seed)

x_batch, y_batch = next(iter(train_loader))
print("x_batch:", x_batch.shape)
print("y_batch:", y_batch.shape)

vocab_size: 2504
x_batch: torch.Size([64, 400])
y_batch: torch.Size([64])


## 6. Định nghĩa mô hình Multi-channel CNN + LSTM

In [8]:
class MultiChannelCnnLstm(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_classes: int,
        embedding_dim: int = 200,
        num_filters: int = 150,
        kernel_sizes: Tuple[int, ...] = (3, 5, 7),  
        lstm_hidden: int = 128,
        dense_hidden: int = 200,
        dropout: float = 0.2,
        pad_idx: int = 0,
    ) -> None:
        super().__init__()

        # Keras gốc dùng 2 Embedding layer riêng
        self.embedding_cnn = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=pad_idx,
        )

        self.embedding_lstm = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=pad_idx,
        )

        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=embedding_dim,
                out_channels=num_filters,
                kernel_size=k,
            )
            for k in kernel_sizes
        ])

        self.relu = nn.ReLU()

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=lstm_hidden,
            batch_first=True,
            bidirectional=False,
        )

        concat_dim = len(kernel_sizes) * num_filters + lstm_hidden

        self.fc = nn.Linear(concat_dim, dense_hidden)
        self.fc_activation = nn.Sigmoid()
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(dense_hidden, num_classes)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        # CNN channel
        emb_cnn = self.embedding_cnn(input_ids)
        cnn_in = emb_cnn.transpose(1, 2)

        pooled_outputs = []
        for conv in self.convs:
            feat = self.relu(conv(cnn_in))
            pooled = torch.max(feat, dim=2).values
            pooled_outputs.append(pooled)

        cnn_features = torch.cat(pooled_outputs, dim=1)

        # LSTM channel
        emb_lstm = self.embedding_lstm(input_ids)
        _, (h_n, _) = self.lstm(emb_lstm)
        lstm_features = h_n[-1]

        # Fusion + classifier
        features = torch.cat([lstm_features, cnn_features], dim=1)

        x = self.fc_activation(self.fc(features))
        x = self.dropout(x)

        # Return logits. Use nn.CrossEntropyLoss during training.
        logits = self.classifier(x)
        return logits

In [9]:
num_classes = 3 if cfg.task == "sentiment" else 4


def get_safe_device(min_cuda_major: int = 7) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")

    try:
        name = torch.cuda.get_device_name(0)
        major, minor = torch.cuda.get_device_capability(0)
        print(f"Detected GPU: {name}, capability: sm_{major}{minor}")

        if major < min_cuda_major:
            print("GPU không tương thích với PyTorch CUDA hiện tại. Fallback sang CPU.")
            return torch.device("cpu")

        # Smoke test nhỏ để bắt lỗi cudaErrorNoKernelImageForDevice sớm.
        _ = torch.empty(1, device="cuda") + 1
        return torch.device("cuda")
    except Exception as exc:
        print("CUDA không dùng được trong runtime này. Fallback sang CPU.")
        print("Reason:", repr(exc))
        return torch.device("cpu")


device = get_safe_device()
seed_everything(cfg.seed)

model = MultiChannelCnnLstm(
    vocab_size=len(vocab),
    num_classes=num_classes,
    embedding_dim=cfg.embedding_dim,
    num_filters=cfg.num_filters,
    kernel_sizes=cfg.kernel_sizes,
    lstm_hidden=cfg.lstm_hidden,
    dense_hidden=cfg.dense_hidden,
    dropout=cfg.dropout,
    pad_idx=vocab[PAD_TOKEN],
).to(device)

with torch.no_grad():
    logits = model(x_batch[:4].to(device))

print(model)
print("device:", device)
print("logits shape:", logits.shape)

Detected GPU: Tesla T4, capability: sm_75
MultiChannelCnnLstm(
  (embedding_cnn): Embedding(2504, 200, padding_idx=0)
  (embedding_lstm): Embedding(2504, 200, padding_idx=0)
  (convs): ModuleList(
    (0): Conv1d(200, 150, kernel_size=(3,), stride=(1,))
    (1): Conv1d(200, 150, kernel_size=(5,), stride=(1,))
    (2): Conv1d(200, 150, kernel_size=(7,), stride=(1,))
  )
  (relu): ReLU()
  (lstm): LSTM(200, 128, batch_first=True)
  (fc): Linear(in_features=578, out_features=200, bias=True)
  (fc_activation): Sigmoid()
  (dropout): Dropout(p=0.2, inplace=False)
  (classifier): Linear(in_features=200, out_features=3, bias=True)
)
device: cuda
logits shape: torch.Size([4, 3])


## 7. Hàm evaluate và train

In [10]:
def make_class_weights(labels: Sequence[int], num_classes: int, device: torch.device) -> torch.Tensor:
    counts = Counter(int(x) for x in labels)
    total = sum(counts.values())
    weights = []
    for c in range(num_classes):
        count_c = max(1, counts.get(c, 0))
        weights.append(total / (num_classes * count_c))
    return torch.tensor(weights, dtype=torch.float32, device=device)


def make_criterion(cfg: TrainConfig, device: torch.device) -> nn.Module:
    if cfg.use_class_weights:
        weights = make_class_weights(train_labels, num_classes, device)
        print("Using class weights:", weights.detach().cpu().numpy().round(4).tolist())
        return nn.CrossEntropyLoss(weight=weights)
    return nn.CrossEntropyLoss()


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device):
    model.eval()
    criterion = make_criterion(cfg, device)
    total_loss = 0.0
    all_preds = []
    all_labels = []

    for input_ids, labels in loader:
        input_ids = input_ids.to(device)
        labels = labels.to(device)
        logits = model(input_ids)
        loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    return {
        "loss": total_loss / max(1, len(all_labels)),
        "accuracy": accuracy_score(all_labels, all_preds),
        "macro_f1": f1_score(all_labels, all_preds, average="macro"),
        "labels": all_labels,
        "preds": all_preds,
    }


def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    cfg: TrainConfig,
    device: torch.device,
):
    output_dir = Path(cfg.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    criterion = make_criterion(cfg, device)
    optimizer = torch.optim.Adamax(model.parameters(), lr=cfg.lr)

    best_macro_f1 = -1.0
    best_path = output_dir / f"best_{cfg.task}_cnn_lstm.pt"
    history = []

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"epoch {epoch}/{cfg.epochs}")

        for input_ids, labels in pbar:
            input_ids = input_ids.to(device)
            labels = labels.to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(input_ids)
            loss = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        train_loss = running_loss / len(train_loader.dataset)
        val_metrics = evaluate(model, val_loader, device)

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
        }
        history.append(row)

        print(
            f"epoch={epoch} "
            f"train_loss={train_loss:.4f} "
            f"val_loss={val_metrics['loss']:.4f} "
            f"val_acc={val_metrics['accuracy']:.4f} "
            f"val_macro_f1={val_metrics['macro_f1']:.4f}"
        )

        if val_metrics["macro_f1"] > best_macro_f1:
            best_macro_f1 = float(val_metrics["macro_f1"])
            torch.save(model.state_dict(), best_path)
            print("saved best checkpoint ->", best_path)

    return history, best_path

## 8. Huấn luyện

In [11]:
seed_everything(cfg.seed)
train_loader, val_loader, test_loader = make_loaders(cfg.batch_size, cfg.seed)

history, best_path = train_model(model, train_loader, val_loader, cfg, device)
history[-3:]

epoch 1/14:   0%|          | 0/179 [00:00<?, ?it/s]

epoch=1 train_loss=0.4192 val_loss=0.3097 val_acc=0.8920 val_macro_f1=0.6172
saved best checkpoint -> outputs_sentiment_cnn_lstm/best_sentiment_cnn_lstm.pt


epoch 2/14:   0%|          | 0/179 [00:00<?, ?it/s]

epoch=2 train_loss=0.2163 val_loss=0.2780 val_acc=0.9059 val_macro_f1=0.7583
saved best checkpoint -> outputs_sentiment_cnn_lstm/best_sentiment_cnn_lstm.pt


epoch 3/14:   0%|          | 0/179 [00:00<?, ?it/s]

...

epoch 13/14:   0%|          | 0/179 [00:00<?, ?it/s]

epoch=13 train_loss=0.0024 val_loss=0.4359 val_acc=0.9090 val_macro_f1=0.7628


epoch 14/14:   0%|          | 0/179 [00:00<?, ?it/s]

epoch=14 train_loss=0.0026 val_loss=0.4456 val_acc=0.9128 val_macro_f1=0.7827


[{'epoch': 12,
  'train_loss': 0.00306091623035141,
  'val_loss': 0.41101174531898693,
  'val_accuracy': 0.9090334807327859,
  'val_macro_f1': 0.7717294406815097},
 {'epoch': 13,
  'train_loss': 0.0024070481051808737,
  'val_loss': 0.4359499012377167,
  'val_accuracy': 0.9090334807327859,
  'val_macro_f1': 0.7627590175135758},
 {'epoch': 14,
  'train_loss': 0.0025615897170648974,
  'val_loss': 0.4455714689841846,
  'val_accuracy': 0.9128237523689198,
  'val_macro_f1': 0.7826927520587391}]

## 9. Đánh giá trên test set

In [12]:
model.load_state_dict(torch.load(best_path, map_location=device))
test_metrics = evaluate(model, test_loader, device)

print("TEST")
print(f"accuracy={test_metrics['accuracy']:.4f}")
print(f"macro_f1={test_metrics['macro_f1']:.4f}")
print()
print(classification_report(test_metrics["labels"], test_metrics["preds"], digits=4))
print("confusion_matrix:")
print(confusion_matrix(test_metrics["labels"], test_metrics["preds"]))

TEST
accuracy=0.8913
macro_f1=0.7465

              precision    recall  f1-score   support

           0     0.8818    0.9368    0.9085      1409
           1     0.5088    0.3473    0.4128       167
           2     0.9286    0.9082    0.9183      1590

    accuracy                         0.8913      3166
   macro avg     0.7731    0.7308    0.7465      3166
weighted avg     0.8856    0.8913    0.8873      3166

confusion_matrix:
[[1320   27   62]
 [  60   58   49]
 [ 117   29 1444]]


## 10. So sánh với các phương pháp khác

Phần này tái hiện tinh thần thí nghiệm trong paper: không chỉ train mô hình đề xuất, mà còn so sánh với các baseline riêng lẻ và baseline truyền thống.

Các nhóm được so sánh:

- **TF-IDF + Logistic Regression**
- **TF-IDF + Linear SVM**
- **TF-IDF + Multinomial Naive Bayes**
- **CNN-only**: chỉ giữ nhánh CNN multi-filter.
- **LSTM-only**: chỉ giữ nhánh LSTM.
- **Multi-channel CNN+LSTM**: mô hình chính đã train ở trên.

Metric chính nên dùng: `macro_f1`, vì sentiment dataset thường lệch nhãn; `accuracy` vẫn được giữ để dễ đối chiếu.


In [13]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC


BASE_METHOD = "Multi-channel CNN+LSTM"


def summarize_predictions(method: str, family: str, y_true, y_pred):
    return {
        "method": method,
        "family": family,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    }


comparison_rows = []

# Thêm kết quả của mô hình chính đã train ở trên.
comparison_rows.append({
    "method": BASE_METHOD,
    "family": "Deep learning / base model",
    "accuracy": test_metrics["accuracy"],
    "macro_f1": test_metrics["macro_f1"],
    "weighted_f1": f1_score(test_metrics["labels"], test_metrics["preds"], average="weighted"),
})

comparison_rows


[{'method': 'Multi-channel CNN+LSTM',
  'family': 'Deep learning / base model',
  'accuracy': 0.8913455464308275,
  'macro_f1': 0.7465198736978907,
  'weighted_f1': 0.8872511002319753}]

### 10.1. Baseline truyền thống: TF-IDF + ML classifier

Các baseline này thường chạy nhanh, là mốc rất tốt để kiểm tra xem deep model có thực sự đáng dùng hay không. Với dataset nhỏ/vừa, SVM hoặc Logistic Regression đôi khi rất cạnh tranh.


In [14]:
classical_models = {
    "TF-IDF + Logistic Regression": LogisticRegression(max_iter=1000, n_jobs=None),
    "TF-IDF + Linear SVM": LinearSVC(),
    "TF-IDF + MultinomialNB": MultinomialNB(),
}

for method_name, classifier in classical_models.items():
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(
            tokenizer=basic_vi_tokenize,
            token_pattern=None,
            lowercase=False,
            ngram_range=(1, 2),
            min_df=2,
            max_features=50000,
        )),
        ("clf", classifier),
    ])
    pipe.fit(train_texts, train_labels)
    preds = pipe.predict(test_texts)
    row = summarize_predictions(method_name, "TF-IDF baseline", test_labels, preds)
    comparison_rows.append(row)
    print(row)


{'method': 'TF-IDF + Logistic Regression', 'family': 'TF-IDF baseline', 'accuracy': 0.8910296904611498, 'macro_f1': 0.6684744341392718, 'weighted_f1': 0.8745003849718849}
{'method': 'TF-IDF + Linear SVM', 'family': 'TF-IDF baseline', 'accuracy': 0.8973468098547063, 'macro_f1': 0.7141728950091432, 'weighted_f1': 0.8866885599859938}
{'method': 'TF-IDF + MultinomialNB', 'family': 'TF-IDF baseline', 'accuracy': 0.861339229311434, 'macro_f1': 0.5897543830087489, 'weighted_f1': 0.8385362483650136}


### 10.2. Deep learning ablation: CNN-only và LSTM-only

Paper nhấn mạnh ý tưởng kết hợp ưu điểm của CNN và LSTM. Vì vậy ta thêm hai ablation quan trọng:

- **CNN-only** kiểm tra sức mạnh của local n-gram features.
- **LSTM-only** kiểm tra sức mạnh của sequential/context features.

Để so sánh công bằng hơn, để `COMPARISON_EPOCHS = cfg.epochs`. Nếu chỉ muốn smoke test nhanh, giảm xuống `1`, `2` hoặc `3`.

In [15]:
class TextCNNOnly(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_classes: int,
        embedding_dim: int = 200,
        num_filters: int = 150,
        kernel_sizes: Tuple[int, ...] = (3, 5, 7),
        dense_hidden: int = 200,
        dropout: float = 0.2,
        pad_idx: int = 0,
    ) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.convs = nn.ModuleList([
            nn.Conv1d(embedding_dim, num_filters, kernel_size=k)
            for k in kernel_sizes
        ])
        self.relu = nn.ReLU()
        concat_dim = len(kernel_sizes) * num_filters
        self.fc = nn.Linear(concat_dim, dense_hidden)
        self.fc_activation = nn.Sigmoid()
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(dense_hidden, num_classes)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        emb = self.embedding(input_ids)
        cnn_in = emb.transpose(1, 2)
        pooled_outputs = []
        for conv in self.convs:
            feat = self.relu(conv(cnn_in))
            pooled_outputs.append(torch.max(feat, dim=2).values)
        features = torch.cat(pooled_outputs, dim=1)
        x = self.fc_activation(self.fc(features))
        x = self.dropout(x)
        return self.classifier(x)


class LSTMOnly(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_classes: int,
        embedding_dim: int = 200,
        lstm_hidden: int = 128,
        dense_hidden: int = 200,
        dropout: float = 0.2,
        pad_idx: int = 0,
    ) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=lstm_hidden,
            batch_first=True,
            bidirectional=False,
        )
        self.fc = nn.Linear(lstm_hidden, dense_hidden)
        self.fc_activation = nn.Sigmoid()
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(dense_hidden, num_classes)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        emb = self.embedding(input_ids)
        _, (h_n, _) = self.lstm(emb)
        features = h_n[-1]
        x = self.fc_activation(self.fc(features))
        x = self.dropout(x)
        return self.classifier(x)

In [16]:
# Đặt False nếu chỉ muốn chạy baseline TF-IDF nhanh.
RUN_DEEP_COMPARISON = True
COMPARISON_EPOCHS = cfg.epochs  # đổi thành 1-3 nếu chỉ smoke test


def build_multi_channel_model() -> nn.Module:
    return MultiChannelCnnLstm(
        vocab_size=len(vocab),
        num_classes=num_classes,
        embedding_dim=cfg.embedding_dim,
        num_filters=cfg.num_filters,
        kernel_sizes=cfg.kernel_sizes,
        lstm_hidden=cfg.lstm_hidden,
        dense_hidden=cfg.dense_hidden,
        dropout=cfg.dropout,
        pad_idx=vocab[PAD_TOKEN],
    )


def build_cnn_only_model() -> nn.Module:
    return TextCNNOnly(
        vocab_size=len(vocab),
        num_classes=num_classes,
        embedding_dim=cfg.embedding_dim,
        num_filters=cfg.num_filters,
        kernel_sizes=cfg.kernel_sizes,
        dense_hidden=cfg.dense_hidden,
        dropout=cfg.dropout,
        pad_idx=vocab[PAD_TOKEN],
    )


def build_lstm_only_model() -> nn.Module:
    return LSTMOnly(
        vocab_size=len(vocab),
        num_classes=num_classes,
        embedding_dim=cfg.embedding_dim,
        lstm_hidden=cfg.lstm_hidden,
        dense_hidden=cfg.dense_hidden,
        dropout=cfg.dropout,
        pad_idx=vocab[PAD_TOKEN],
    )


def train_and_evaluate_deep_baseline(method_name: str, build_model_fn, run_seed: int = None):
    """
    Quan trọng: model phải được khởi tạo SAU khi seed được set.
    Nếu truyền sẵn model đã khởi tạo, kết quả comparison sẽ không còn thật sự reproducible.
    """
    if run_seed is None:
        run_seed = cfg.seed

    seed_everything(run_seed)
    local_train_loader, local_val_loader, local_test_loader = make_loaders(cfg.batch_size, run_seed)

    model_local = build_model_fn().to(device)

    baseline_cfg = TrainConfig(**asdict(cfg))
    baseline_cfg.epochs = COMPARISON_EPOCHS
    baseline_cfg.seed = run_seed

    safe_name = method_name.lower().replace(" ", "_").replace("+", "plus").replace("-", "_")
    baseline_cfg.output_dir = str(Path(cfg.output_dir) / "comparison" / safe_name / f"seed_{run_seed}")

    history_baseline, best_path_baseline = train_model(
        model_local,
        local_train_loader,
        local_val_loader,
        baseline_cfg,
        device,
    )

    model_local.load_state_dict(torch.load(best_path_baseline, map_location=device))
    metrics = evaluate(model_local, local_test_loader, device)

    row = {
        "method": method_name,
        "family": "Deep learning ablation",
        "seed": run_seed,
        "accuracy": metrics["accuracy"],
        "macro_f1": metrics["macro_f1"],
        "weighted_f1": f1_score(metrics["labels"], metrics["preds"], average="weighted"),
    }
    return row, history_baseline, metrics


if RUN_DEEP_COMPARISON:
    cnn_row, cnn_history, cnn_test_metrics = train_and_evaluate_deep_baseline(
        "CNN-only",
        build_cnn_only_model,
        run_seed=cfg.seed,
    )
    comparison_rows.append(cnn_row)
    print(cnn_row)

    lstm_row, lstm_history, lstm_test_metrics = train_and_evaluate_deep_baseline(
        "LSTM-only",
        build_lstm_only_model,
        run_seed=cfg.seed,
    )
    comparison_rows.append(lstm_row)
    print(lstm_row)
else:
    print("Skipped deep comparison. Set RUN_DEEP_COMPARISON = True to run CNN-only and LSTM-only.")

epoch 1/14:   0%|          | 0/179 [00:00<?, ?it/s]

epoch=1 train_loss=0.4069 val_loss=0.3010 val_acc=0.8964 val_macro_f1=0.6288
saved best checkpoint -> outputs_sentiment_cnn_lstm/comparison/cnn_only/seed_1337/best_sentiment_cnn_lstm.pt


epoch 2/14:   0%|          | 0/179 [00:00<?, ?it/s]

epoch=2 train_loss=0.2058 val_loss=0.2842 val_acc=0.9052 val_macro_f1=0.7687
saved best checkpoint -> outputs_sentiment_cnn_lstm/comparison/cnn_only/seed_1337/best_sentiment_cnn_lstm.pt


epoch 3/14:   0%|          | 0/179 [00:00<?, ?it/s]

...

epoch=12 train_loss=0.8358 val_loss=0.8472 val_acc=0.5085 val_macro_f1=0.2247


epoch 13/14:   0%|          | 0/179 [00:00<?, ?it/s]

epoch=13 train_loss=0.8341 val_loss=0.8464 val_acc=0.5085 val_macro_f1=0.2247


epoch 14/14:   0%|          | 0/179 [00:00<?, ?it/s]

epoch=14 train_loss=0.8350 val_loss=0.8524 val_acc=0.4454 val_macro_f1=0.2054
{'method': 'LSTM-only', 'family': 'Deep learning ablation', 'seed': 1337, 'accuracy': 0.5022109917877448, 'macro_f1': 0.22287636669470143, 'weighted_f1': 0.33579288349138525}


### 10.3. Bảng tổng hợp kết quả

Bảng dưới đây là kết quả thực nghiệm trên cùng split train/validation/test. Khi viết báo cáo, nên trích bảng này và ghi rõ cấu hình tokenizer, max length, epoch, batch size, random seed.


In [17]:
comparison_df = pd.DataFrame(comparison_rows)

# Một số row cũ có thể chưa có cột seed, ví dụ TF-IDF hoặc base model.
if "seed" not in comparison_df.columns:
    comparison_df["seed"] = np.nan

base_macro = comparison_df.loc[comparison_df["method"] == BASE_METHOD, "macro_f1"].iloc[0]
base_acc = comparison_df.loc[comparison_df["method"] == BASE_METHOD, "accuracy"].iloc[0]

comparison_df["delta_macro_f1_vs_base"] = comparison_df["macro_f1"] - base_macro
comparison_df["delta_accuracy_vs_base"] = comparison_df["accuracy"] - base_acc
comparison_df["is_base_model"] = comparison_df["method"].eq(BASE_METHOD)

comparison_df = comparison_df.sort_values("macro_f1", ascending=False).reset_index(drop=True)
comparison_df["rank_by_macro_f1"] = np.arange(1, len(comparison_df) + 1)

# Làm tròn để bảng dễ đọc.
display(comparison_df.style.format({
    "accuracy": "{:.4f}",
    "macro_f1": "{:.4f}",
    "weighted_f1": "{:.4f}",
    "delta_macro_f1_vs_base": "{:+.4f}",
    "delta_accuracy_vs_base": "{:+.4f}",
}))

comparison_out = Path(cfg.output_dir) / "comparison_results.csv"
comparison_out.parent.mkdir(parents=True, exist_ok=True)
comparison_df.to_csv(comparison_out, index=False)
print("saved comparison table ->", comparison_out.resolve())


def explain_if_base_is_not_best(df: pd.DataFrame, base_method: str = BASE_METHOD, metric: str = "macro_f1"):
    best = df.iloc[0]
    base = df[df["method"] == base_method].iloc[0]
    diff = float(best[metric] - base[metric])

    print("-" * 80)

    if best["method"] == base_method:
        print(f"Base model đang đứng đầu theo {metric}. Có thể báo cáo đây là model tốt nhất trong lần chạy này.")
        return

    print(f"Base model KHÔNG đứng đầu theo {metric} trong lần chạy này.")
    print(f"Best method: {best['method']} | {metric}={best[metric]:.4f}")
    print(f"Base method: {base_method} | {metric}={base[metric]:.4f}")
    print(f"Chênh lệch: {diff:+.4f}")

    if abs(diff) < 0.005:
        print("Chênh lệch rất nhỏ (<0.5 điểm %).")
    elif abs(diff) < 0.01:
        print("Chênh lệch nhỏ (<1 điểm %).")
    else:
        print("Chênh lệch đáng kể hơn. Nếu kết quả này lặp lại qua nhiều seed.")


explain_if_base_is_not_best(comparison_df)


,method,family,accuracy,macro_f1,weighted_f1,seed,delta_macro_f1_vs_base,delta_accuracy_vs_base,is_base_model,rank_by_macro_f1
0,CNN-only,Deep learning ablation,0.8923,0.7537,0.8883,1337.000000,+0.0072,+0.0009,False,1
1,Multi-channel CNN+LSTM,Deep learning / base model,0.8913,0.7465,0.8873,nan,+0.0000,+0.0000,True,2
2,TF-IDF + Linear SVM,TF-IDF baseline,0.8973,0.7142,0.8867,nan,-0.0323,+0.0060,False,3
3,TF-IDF + Logistic Regression,TF-IDF baseline,0.8910,0.6685,0.8745,nan,-0.0780,-0.0003,False,4
4,TF-IDF + MultinomialNB,TF-IDF baseline,0.8613,0.5898,0.8385,nan,-0.1568,-0.0300,False,5
5,LSTM-only,Deep learning ablation,0.5022,0.2229,0.3358,1337.000000,-0.5236,-0.3891,False,6


saved comparison table -> /kaggle/working/outputs_sentiment_cnn_lstm/comparison_results.csv
--------------------------------------------------------------------------------
Base model KHÔNG đứng đầu theo macro_f1 trong lần chạy này.
Best method: CNN-only | macro_f1=0.7537
Base method: Multi-channel CNN+LSTM | macro_f1=0.7465
Chênh lệch: +0.0072
Chênh lệch nhỏ (<1 điểm %).


## 11. Lưu config, vocab và report


In [18]:
output_dir = Path(cfg.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump(asdict(cfg) | {"vocab_size": len(vocab), "num_classes": num_classes}, f, indent=2, ensure_ascii=False)

with open(output_dir / "vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False)

report = classification_report(test_metrics["labels"], test_metrics["preds"], digits=4)
with open(output_dir / "test_report.txt", "w", encoding="utf-8") as f:
    f.write(f"accuracy={test_metrics['accuracy']:.4f}\n")
    f.write(f"macro_f1={test_metrics['macro_f1']:.4f}\n\n")
    f.write(report)

print("saved to", output_dir.resolve())

saved to /kaggle/working/outputs_sentiment_cnn_lstm


## 12. Tổng hợp kết quả và nhận xét

### 12.1. Bảng kết quả thực nghiệm

| Phương pháp                      | Accuracy | Macro-F1 | Weighted-F1 | Nhận xét ngắn                                                                                                                 |
| -------------------------------- | -------: | -------: | ----------: | ----------------------------------------------------------------------------------------------------------------------------- |
| TF-IDF + Logistic Regression     |   0.8910 |   0.6685 |      0.8745 | Baseline tuyến tính đơn giản, cho accuracy khá tốt nhưng Macro-F1 còn thấp so với các mô hình học sâu.                        |
| TF-IDF + Linear SVM              |   0.8973 |   0.7142 |      0.8867 | Đạt accuracy cao nhất trong các phương pháp thử nghiệm, cho thấy TF-IDF vẫn là baseline mạnh cho bài toán này.                |
| TF-IDF + Multinomial Naive Bayes |   0.8613 |   0.5898 |      0.8385 | Chạy nhanh và đơn giản, nhưng kết quả thấp hơn các baseline còn lại.                                                          |
| CNN-only                         |   0.8923 |   0.7537 |      0.8883 | Đạt Macro-F1 cao nhất, cho thấy các đặc trưng cục bộ như cụm từ/ngữ cảnh ngắn có vai trò quan trọng trong bài toán sentiment. |
| LSTM-only                        |   0.5022 |   0.2229 |      0.3358 | Kết quả thấp, cho thấy mô hình LSTM-only chưa học tốt trong thiết lập thí nghiệm này.                                         |
| Multi-channel CNN + LSTM         |   0.8913 |   0.7465 |      0.8873 | Mô hình chính đạt kết quả tốt và gần với CNN-only, nhưng chưa vượt CNN-only trên tập dữ liệu này.                             |

### 12.2. Nhận xét theo kết quả

| Nội dung                                 | Kết quả quan sát được                                                                              | Nhận xét                                                                                                                                                                   |
| ---------------------------------------- | -------------------------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| Mô hình tốt nhất theo Macro-F1           | CNN-only đạt Macro-F1 cao nhất: 0.7537.                                                            | Macro-F1 phù hợp để đánh giá khi dữ liệu có thể lệch nhãn, nên CNN-only là mô hình nổi bật nhất theo tiêu chí này.                                                         |
| Mô hình tốt nhất theo Accuracy           | TF-IDF + Linear SVM đạt Accuracy cao nhất: 0.8973.                                                 | Điều này cho thấy baseline truyền thống vẫn rất cạnh tranh, đặc biệt khi dữ liệu có tín hiệu từ vựng rõ ràng.                                                              |
| CNN-only so với Multi-channel CNN + LSTM | CNN-only nhỉnh hơn Multi-channel CNN + LSTM về Macro-F1 và Accuracy.                               | Kết quả này hợp lý vì các câu phản hồi sinh viên thường ngắn, trong đó cụm từ mang cảm xúc có thể đã đủ để phân loại tốt.                                                  |
| LSTM-only so với các mô hình khác        | LSTM-only có kết quả thấp nhất.                                                                    | Có thể mô hình chưa hội tụ tốt hoặc chưa phù hợp với thiết lập hiện tại. Ngoài ra, tín hiệu cảm xúc trong dataset có thể nằm nhiều ở cụm từ ngắn hơn là phụ thuộc dài hạn. |
| So với bài báo gốc                       | Bài báo gốc báo cáo mô hình Multi-channel CNN + LSTM tốt hơn CNN và LSTM trên các dataset VS/VLSP. | Kết quả ở đây không nhất thiết phải trùng với bài báo do khác dataset, preprocessing, framework và cách huấn luyện.                                                        |

### 12.3. Kết luận

Trong thí nghiệm trên dataset UIT-VSFC, các baseline truyền thống và mô hình học sâu đều đạt kết quả tương đối tốt cho bài toán phân loại cảm xúc tiếng Việt. TF-IDF + Linear SVM đạt accuracy cao nhất, trong khi CNN-only đạt Macro-F1 cao nhất.

Mô hình Multi-channel CNN + LSTM là mô hình chính của bài, được xây dựng dựa trên hướng tiếp cận trong bài báo gốc. Kết quả của mô hình này khá gần với CNN-only, nhưng chưa tạo ra cải thiện rõ ràng trên dataset UIT-VSFC. Điều này cho thấy việc kết hợp CNN và LSTM không luôn đảm bảo tốt hơn mô hình đơn giản hơn, đặc biệt khi dữ liệu gồm nhiều câu ngắn và tín hiệu cảm xúc nằm chủ yếu ở các cụm từ cục bộ.

Kết quả hiện tại nên được xem là kết quả thực nghiệm trong một thiết lập cụ thể. Để kết luận chắc hơn, có thể chạy thêm nhiều seed và báo cáo giá trị trung bình cùng độ lệch chuẩn.


## 13. Inference thử với câu mới


In [19]:
label_names = {
    "sentiment": ["negative", "neutral", "positive"],
    "topic": ["lecturer", "training_program", "facility", "others"],
}

@torch.no_grad()
def predict(texts: List[str]):
    model.eval()
    input_ids = torch.tensor([encode_text(t, vocab, cfg.max_len) for t in texts], dtype=torch.long).to(device)
    logits = model(input_ids)
    probs = torch.softmax(logits, dim=1).cpu().numpy()
    preds = probs.argmax(axis=1)
    results = []
    for text, pred, prob in zip(texts, preds, probs):
        results.append({
            "text": text,
            "pred_id": int(pred),
            "pred_label": label_names[cfg.task][int(pred)],
            "confidence": float(prob[int(pred)]),
        })
    return results

samples = [
    "giảng viên dạy rất dễ hiểu và nhiệt tình .",
    "phòng học nóng và thiết bị quá cũ .",
    "em chưa có ý kiến về môn học này .",
]

predict(samples)

[{'text': 'giảng viên dạy rất dễ hiểu và nhiệt tình .',
  'pred_id': 2,
  'pred_label': 'positive',
  'confidence': 0.9999713897705078},
 {'text': 'phòng học nóng và thiết bị quá cũ .',
  'pred_id': 0,
  'pred_label': 'negative',
  'confidence': 0.9998950958251953},
 {'text': 'em chưa có ý kiến về môn học này .',
  'pred_id': 1,
  'pred_label': 'neutral',
  'confidence': 0.9104200005531311}]